# Experiment Significance Analysis

**Question:** Does moving the first gate from level 30 (`gate_30`, control) to level 40 (`gate_40`, treatment) change player retention?

This notebook queries the `fct_experiment_metrics` mart directly from BigQuery, runs a two-proportion z-test on Day-1 and Day-7 retention, computes 95% confidence intervals for the difference, and translates the result into an estimated revenue impact.

**Data:** Cookie Cats A/B test dataset (Kaggle) for real experiment data; a small synthetic accounts table exists separately for account-level dimensions (industry, ARR tier) and is not used in this notebook's statistical test, since retention data is at the player level.

In [11]:
import os
from dotenv import load_dotenv
from google.cloud import bigquery
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

load_dotenv()

PROJECT_ID = os.environ["GCP_PROJECT_ID"]
DATASET = os.environ["BQ_DATASET"]

client = bigquery.Client(project=PROJECT_ID)

## 1. Pull the underlying player-level data

The mart is pre-aggregated by variant and engagement tier, which is right for dashboarding, but a proportion test needs raw success/failure counts per variant (not sliced by tier) to test the overall experiment result. So we query the mart and aggregate up one level here, in Python, to keep the test itself simple and transparent.

In [12]:
query = f"""
    SELECT
        experiment_variant,
        SUM(total_players) AS total_players,
        SUM(retained_day_1_count) AS retained_day_1_count,
        SUM(retained_day_7_count) AS retained_day_7_count
    FROM `{PROJECT_ID}.{DATASET}.fct_experiment_metrics`
    GROUP BY experiment_variant
    ORDER BY experiment_variant
"""

summary = client.query(query).to_dataframe()
summary

,experiment_variant,total_players,retained_day_1_count,retained_day_7_count
0,gate_30,44700,20034,8502
1,gate_40,45489,20119,8279


## 2. Two-proportion z-test — Day 1 retention

**Why a z-test, not a t-test:** retention is a binary outcome (returned / didn't return), so we're comparing two proportions, not two means. A two-proportion z-test (or equivalently a chi-square test) is the correct choice here. A t-test would be appropriate if we were comparing a continuous metric like average game rounds instead.

In [13]:
control = summary[summary.experiment_variant == 'gate_30'].iloc[0]
treatment = summary[summary.experiment_variant == 'gate_40'].iloc[0]

def run_proportion_test(control, treatment, count_col, label):
    counts = np.array([control[count_col], treatment[count_col]])
    nobs = np.array([control['total_players'], treatment['total_players']])

    z_stat, p_value = proportions_ztest(counts, nobs)

    rate_control = counts[0] / nobs[0]
    rate_treatment = counts[1] / nobs[1]
    diff = rate_treatment - rate_control

    ci_low, ci_high = confint_proportions_2indep(
        count1=counts[1], nobs1=nobs[1],
        count2=counts[0], nobs2=nobs[0],
        method='wald'
    )

    print(f"--- {label} ---")
    print(f"Control (gate_30) rate:   {rate_control:.4%}")
    print(f"Treatment (gate_40) rate: {rate_treatment:.4%}")
    print(f"Absolute difference:      {diff:.4%}")
    print(f"95% CI for difference:    [{ci_low:.4%}, {ci_high:.4%}]")
    print(f"z-statistic: {z_stat:.4f}")
    print(f"p-value:     {p_value:.4f}")
    if p_value < 0.05:
        print("Result: statistically significant at the 95% confidence level.")
    else:
        print("Result: NOT statistically significant at the 95% confidence level.")
    print()

    return {
        'metric': label,
        'rate_control': rate_control,
        'rate_treatment': rate_treatment,
        'diff': diff,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'p_value': p_value,
    }

day1_result = run_proportion_test(control, treatment, 'retained_day_1_count', 'Day-1 Retention')

--- Day-1 Retention ---
Control (gate_30) rate:   44.8188%
Treatment (gate_40) rate: 44.2283%
Absolute difference:      -0.5905%
95% CI for difference:    [-1.2392%, 0.0582%]
z-statistic: 1.7841
p-value:     0.0744
Result: NOT statistically significant at the 95% confidence level.



## 3. Two-proportion z-test — Day 7 retention

In [14]:
day7_result = run_proportion_test(control, treatment, 'retained_day_7_count', 'Day-7 Retention')

--- Day-7 Retention ---
Control (gate_30) rate:   19.0201%
Treatment (gate_40) rate: 18.2000%
Absolute difference:      -0.8201%
95% CI for difference:    [-1.3282%, -0.3121%]
z-statistic: 3.1644
p-value:     0.0016
Result: statistically significant at the 95% confidence level.



## 4. Caveats and assumptions

- **Multiple comparisons:** we ran two significance tests (Day-1 and Day-7) on the same underlying population. Running multiple tests increases the chance of a false positive; a stricter analysis would apply a correction (e.g. Bonferroni) to the significance threshold. Noted here rather than applied, given the small scope of this test.
- **No novelty/seasonality control:** this is a single static dataset with no information on when the experiment ran or for how long, so we can't rule out day-of-week or novelty effects.
- **Practical vs statistical significance:** even a statistically significant difference in retention may be too small to matter for a real product decision — the confidence interval width matters as much as the p-value.
- **Multiple-comparison note:** engagement-tier-level breakdowns (8 tiers total) are available in the mart for further slicing, but running significance tests on every tier without correction would meaningfully inflate false-positive risk, so that is intentionally not done here.

## 5. Value translation: retention lift to estimated revenue impact

This section translates the observed difference into a rough, clearly-labeled estimate of revenue impact for a hypothetical business — the kind of translation a presales/value-consulting analysis would produce. All inputs below are illustrative assumptions, stated explicitly, not derived from the dataset.

In [15]:
# --- Illustrative business assumptions (edit these for a different scenario) ---
monthly_active_users = 100_000
avg_revenue_per_retained_user_per_month = 2.50  # e.g. ad revenue or in-app purchase avg

day7_lift = day7_result['diff']  # can be negative
day7_lift_low, day7_lift_high = day7_result['ci_low'], day7_result['ci_high']

def estimate_monthly_revenue_impact(lift):
    additional_retained_users = monthly_active_users * lift
    return additional_retained_users * avg_revenue_per_retained_user_per_month

point_estimate = estimate_monthly_revenue_impact(day7_lift)
low_estimate = estimate_monthly_revenue_impact(day7_lift_low)
high_estimate = estimate_monthly_revenue_impact(day7_lift_high)

print("Assumptions: 100,000 MAU, $2.50 avg monthly revenue per retained user (illustrative only)")
print(f"Day-7 retention lift (treatment vs control): {day7_lift:.4%}")
print(f"Point estimate of monthly revenue impact:     ${point_estimate:,.0f}")
print(f"95% CI for monthly revenue impact:            [${low_estimate:,.0f}, ${high_estimate:,.0f}]")

Assumptions: 100,000 MAU, $2.50 avg monthly revenue per retained user (illustrative only)
Day-7 retention lift (treatment vs control): -0.8201%
Point estimate of monthly revenue impact:     $-2,050
95% CI for monthly revenue impact:            [$-3,320, $-780]


## 6. Summary

Moving the gate from level 30 to 40 showed a small, directionally negative effect on both 
Day-1 and Day-7 retention, but the effect was only statistically significant at Day-7.

- **Day-1 retention:** -0.59 percentage points (95% CI: [-1.24%, 0.06%], p = 0.074) — 
  not statistically significant at the 95% confidence level, since the interval crosses zero.
- **Day-7 retention:** -0.82 percentage points, statistically significant at the 95% confidence 
  level (interval does not cross zero).

The effect appears to strengthen over the first week rather than showing up immediately — a 
pattern worth flagging to stakeholders, since a Day-1-only view would have missed it entirely.

Translated into an illustrative revenue model (100,000 MAU, $2.50 avg monthly revenue per 
retained user), the Day-7 effect corresponds to an estimated **-$2,050 per month** 
(95% CI: [-$3,320, -$780]).

This should be read as a directional estimate under illustrative assumptions, not a precise 
forecast of any real product's revenue — see caveats above (multiple comparisons across two 
time horizons, no seasonality control, and the gap between statistical and practical significance).